# Pandas 02 — Cleaning: dtypes, duplicates, gaps and missing values

**What's in here**
- Turning the messy `hourly_power_raw.csv` into a tidy hourly series, step by step
- Timestamps → sorted, tz-aware, unique
- `pd.to_numeric(errors="coerce")`, sentinel replacement, constant columns
- Reindexing to a complete hourly grid and locating the missing hours
- Fill strategies (`ffill`, `interpolate`, group medians) and when each is *not* acceptable
- Outliers: clip vs winsorise vs flag
- String and boolean clean-up on `meters.csv`
- Final `assert` checks and a comparison against the known-good file

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

In [2]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17457 entries, 0 to 17456
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             17457 non-null  object 
 1   consumption_mwh  17457 non-null  float64
 2   temp_c           17308 non-null  float64
 3   wind_ms          17457 non-null  float64
 4   solar_wm2        17457 non-null  float64
 5   price_eur_mwh    17457 non-null  object 
 6   region           17457 non-null  object 
dtypes: float64(4), object(3)
memory usage: 954.8+ KB


## 1. Timestamps first

Convert, then **sort**. Almost every time-series operation (`shift`, `diff`, `rolling`, merges with `asof`) silently assumes the rows are in time order.

In [3]:
df = raw.copy()
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
print("unparseable:", df["time"].isna().sum())
df = df.sort_values("time").reset_index(drop=True)
df.head(3)

unparseable: 0


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,GB
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,GB
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,GB


## 2. Duplicates: exact rows vs duplicate keys

`drop_duplicates()` removes rows identical in every column. `drop_duplicates(subset=["time"])` removes rows with the same timestamp even if other values differ — check first whether those disagreeing rows exist, because then you have to *choose* which one to trust.

In [4]:
exact = df.duplicated().sum()
by_key = df.duplicated(subset=["time"]).sum()
print(f"exact duplicates: {exact}, duplicate timestamps: {by_key}")

# duplicates on the key that are NOT exact duplicates -> conflicting information
conflicting = df[df.duplicated(subset=["time"], keep=False) & ~df.duplicated(keep=False)]
print("conflicting duplicate timestamps:", len(conflicting))

exact duplicates: 15, duplicate timestamps: 15
conflicting duplicate timestamps: 0


In [5]:
df = df.drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)
assert df["time"].is_unique
df.shape

(17442, 7)

**Interview check:** *"Why `keep='last'`?"* — in a feed that appends corrections, the later row is usually the corrected one. If you do not know that, say so; the honest answer is "depends on how the file was produced".

## 3. Numeric columns stored as text

`pd.to_numeric(errors="coerce")` turns anything unparseable into NaN. **Always count how many values you just destroyed** and look at what they were.

In [6]:
bad = df.loc[pd.to_numeric(df["price_eur_mwh"], errors="coerce").isna(), "price_eur_mwh"]
print("values that will become NaN:", bad.value_counts(dropna=False).to_dict())

df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
print(df["price_eur_mwh"].dtype, "| NaN now:", df["price_eur_mwh"].isna().sum())

values that will become NaN: {'missing': 100}
float64 | NaN now: 100


## 4. Sentinel values

`describe()` shows `min = -999` for temperature. Replace sentinels with NaN **before** any statistics, plotting or modelling — a handful of `-999`s wrecks a mean and a regression coefficient.

In [7]:
print(df["temp_c"].describe().round(2))
n_sentinel = (df["temp_c"] == -999).sum()
df["temp_c"] = df["temp_c"].replace(-999.0, np.nan)
print(f"\nreplaced {n_sentinel} sentinels; temp NaN now {df['temp_c'].isna().sum()}")

count    17293.00
mean         6.26
std         61.15
min       -999.00
25%          4.45
50%          9.92
75%         15.32
max         27.74
Name: temp_c, dtype: float64

replaced 63 sentinels; temp NaN now 212


**Pitfall:** `.replace(-999, np.nan)` only catches the exact value. Physical range checks are more robust: temperatures below −60 °C or consumption ≤ 0 are impossible here regardless of the sentinel used.

In [8]:
mask_bad = (df["temp_c"] < -60) | (df["temp_c"] > 60)
print("out-of-range temps left:", mask_bad.sum())
print("non-positive consumption:", (df["consumption_mwh"] <= 0).sum())

out-of-range temps left: 0
non-positive consumption: 0


## 5. Drop constant / useless columns

In [9]:
const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("dropping:", const)
df = df.drop(columns=const)

dropping: ['region']


## 6. Reindex to a complete hourly grid

Build the full `date_range` between first and last stamp and `reindex`. Rows that appear are the missing hours (all-NaN). Now missingness is *visible* instead of hidden in a shorter table.

In [10]:
df = df.set_index("time")
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h", tz="UTC")
print(f"rows present: {len(df)}, expected: {len(full_idx)}, missing hours: {len(full_idx) - len(df)}")
df = df.reindex(full_idx)
df.index.name = "time"

rows present: 17442, expected: 17520, missing hours: 78


Which hours are missing, and are they scattered or whole days? Group the missing stamps by date.

In [11]:
missing = df.index[df["consumption_mwh"].isna()]
missing_by_day = pd.Series(1, index=missing).groupby(missing.date).sum().sort_values(ascending=False)
print("days with missing hours:", len(missing_by_day))
missing_by_day.head(5)

days with missing hours: 54


2022-03-27    24
2022-06-07     2
2022-01-07     1
2023-08-11     1
2023-01-27     1
dtype: int64

**Interview check:** *"2022-03-27 is missing entirely — is that random?"* No: it is the European DST change date. Structured gaps often come from a timezone bug in whoever produced the file. That matters for how you fill (or whether you fill at all).

## 7. Missing-value strategies

Rule of thumb:

| column role | acceptable | not acceptable |
|---|---|---|
| **target** you are forecasting | leave NaN, drop those rows from training | `ffill` / interpolation (you would be training on invented targets) |
| slowly-varying **feature** (temperature) | `interpolate(method="time")`, `ffill(limit=k)` | filling a whole missing day |
| **price** (jumpy) | `ffill(limit=1..2)` or leave NaN | linear interpolation across a spike |

Whatever you do, keep a flag column so the model / your future self knows which values are imputed.

In [12]:
df["temp_was_missing"] = df["temp_c"].isna()

# small gaps only: interpolate along time, but never bridge more than 3 hours
df["temp_c_filled"] = df["temp_c"].interpolate(method="time", limit=3, limit_area="inside")
print("temp NaN before/after:", df["temp_c"].isna().sum(), df["temp_c_filled"].isna().sum())

temp NaN before/after: 290 21


In [13]:
# ffill with a limit: prices carried forward for at most 2 hours
df["price_filled"] = df["price_eur_mwh"].ffill(limit=2)
print("price NaN before/after:", df["price_eur_mwh"].isna().sum(), df["price_filled"].isna().sum())

price NaN before/after: 178 22


**Pitfall:** `ffill()` with no limit across the missing day would copy one hour's price 24 times. With time series, `ffill` is also *causal* (uses the past) while `bfill` and `interpolate` are **not** — `interpolate` uses the next observed value, which at prediction time you would not yet have.

## 8. Fill with a group statistic

For diurnal quantities a sensible fallback is the median for that hour of day. `groupby(...).transform("median")` returns a Series aligned to the original index, so it drops straight into `fillna`.

In [14]:
hour_median = df["consumption_mwh"].groupby(df.index.hour).transform("median")
df["consumption_hourmed_fill"] = df["consumption_mwh"].fillna(hour_median)
print(df["consumption_mwh"].isna().sum(), "->", df["consumption_hourmed_fill"].isna().sum())

78 -> 0


## 9. `dropna`: `subset=` vs `how=`

`dropna()` with no arguments drops any row with *any* NaN — usually far too aggressive. `subset=` names the columns that must be present; `how="all"` drops only rows that are entirely empty (our reindexed gaps).

In [15]:
print("rows:", len(df))
print("dropna(how='all') on original cols:", len(df.dropna(how="all", subset=["consumption_mwh", "temp_c", "price_eur_mwh"])))
print("dropna(subset=target):", len(df.dropna(subset=["consumption_mwh"])))
print("dropna() everything:", len(df.dropna()))

rows: 17520
dropna(how='all') on original cols: 17442
dropna(subset=target): 17442
dropna() everything: 17131


## 10. Outliers: clip vs winsorise vs flag

- `clip(lower, upper)` with **fixed physical bounds** — safe (a negative wind speed is a bug).
- Winsorising at quantiles changes real extreme values — in power markets the spikes *are* the story, so be careful.
- Flagging (a boolean column) keeps the data intact and lets the model / analysis decide.

In [16]:
q_lo, q_hi = df["price_eur_mwh"].quantile([0.005, 0.995])
df["price_is_spike"] = df["price_eur_mwh"] > q_hi
df["price_winsor"] = df["price_eur_mwh"].clip(lower=q_lo, upper=q_hi)
df["wind_ms"] = df["wind_ms"].clip(lower=0)       # physical bound
print(f"quantile bounds: {q_lo:.1f} / {q_hi:.1f}, spikes flagged: {df['price_is_spike'].sum()}")
df[["price_eur_mwh", "price_winsor"]].describe().round(1)

quantile bounds: 11.6 / 195.8, spikes flagged: 87


,price_eur_mwh,price_winsor
count,17342.0,17342.0
mean,98.5,98.3
std,36.9,35.7
min,-19.9,11.6
25%,73.5,73.5
50%,97.7,97.7
75%,122.8,122.8
max,419.6,195.8


## 11. Final checks — make them `assert`s

Assertions turn assumptions into things that fail loudly when the next data delivery breaks them.

In [17]:
assert df.index.is_monotonic_increasing
assert df.index.is_unique
assert (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all()
assert len(df) == 24 * 730          # 2022 + 2023
assert df["temp_c"].min() > -60
assert (df["consumption_mwh"].dropna() > 0).all()
print("all checks passed")

all checks passed


## 12. Compare against the known-good file

When a reference exists, diff against it. Merge on the key, compute absolute differences, and look at the maximum — not just "looks similar".

In [18]:
ref = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
cols = ["consumption_mwh", "temp_c", "wind_ms", "solar_wm2", "price_eur_mwh"]
both = df[cols].join(ref[cols], how="inner", lsuffix="_mine", rsuffix="_ref")
diffs = {c: (both[f"{c}_mine"] - both[f"{c}_ref"]).abs().max() for c in cols}
print("rows compared:", len(both))
pd.Series(diffs).round(6)

rows compared: 17520


consumption_mwh    0.0
temp_c             0.0
wind_ms            0.0
solar_wm2          0.0
price_eur_mwh      0.0
dtype: float64

Max abs difference is 0 where both have data — the cleaning reproduced the source. The rows we could not recover (missing hours, coerced prices) are NaN on our side, which is the honest state.

## 13. String clean-up on a tabular file

Typical problems: inconsistent case, stray whitespace, several spellings of one category. Normalise with the `.str` accessor, then map to canonical labels. Check `value_counts()` before *and* after.

In [19]:
meters = pd.read_csv("../data/meters.csv")
print(meters["region"].value_counts().to_dict())
meters["region"] = meters["region"].str.strip().str.title()
print(meters["region"].value_counts().to_dict())

{'London': 96, 'North': 65, 'Scotland': 57, 'Midlands': 44, 'Wales': 32, 'london': 3, 'wales': 1, 'north': 1, 'midlands': 1}
{'London': 99, 'North': 66, 'Scotland': 57, 'Midlands': 45, 'Wales': 33}


In [20]:
tariff_map = {"Fixed": "fixed", "Variable": "variable", "TOU": "time_of_use"}
meters["tariff"] = meters["tariff"].map(tariff_map)          # unmapped -> NaN, deliberately
meters["tariff"].value_counts(dropna=False)

tariff
fixed          143
variable        94
time_of_use     50
NaN             13
Name: count, dtype: int64

**Pitfall:** `Series.map(dict)` returns NaN for anything not in the dict. That is a feature if you want to be strict, a bug if you forgot a spelling. `.replace(dict)` leaves unknown values untouched instead.

## 14. Booleans read as text, dates as text, categoricals

`"True"`/`"False"` strings are not booleans. Dates need `parse_dates` or `to_datetime`. Low-cardinality strings become `category` — smaller and faster for groupby.

In [21]:
m2 = pd.read_csv("../data/meters.csv", dtype={"has_solar": str})
print("read as text:", m2["has_solar"].dtype, m2["has_solar"].unique())
m2["has_solar"] = m2["has_solar"].str.lower().map({"true": True, "false": False})
m2["signup_date"] = pd.to_datetime(m2["signup_date"])
m2["region"] = m2["region"].str.title().astype("category")
m2.dtypes

read as text: object ['False' 'True']


meter_id                       object
region                       category
tariff                         object
customer_type                  object
annual_kwh_estimate           float64
signup_date            datetime64[ns]
has_solar                        bool
dtype: object

In [22]:
print("memory object vs category (bytes):",
      meters["region"].memory_usage(deep=True), "->", m2["region"].memory_usage(deep=True))

memory object vs category (bytes): 19133 -> 917


## The cleaning recipe (order matters)

1. Parse timestamps (`utc=True`, `errors="coerce"`), **sort**.
2. Resolve duplicate keys — count conflicting ones first.
3. Fix dtypes: `to_numeric(errors="coerce")` and count what you coerced.
4. Replace sentinels and impossible values with NaN.
5. Drop constant columns.
6. Reindex to the full grid so gaps are visible.
7. Decide per column: leave NaN / interpolate / ffill(limit) / group fill — never fill the target.
8. Flag outliers rather than deleting them.
9. `assert` the invariants; compare with a reference if one exists.